# Trustworthy Neural Temporal Point Processes
## Enhancing Reliability via Rotary Embeddings and Uncertainty Quantification under Distribution Shift

Este notebook demonstra como transformar um modelo **RoTHP (Rotary Transformer Hawkes Process)** em um sistema de IA confiável ("Trustworthy AI").

Abordamos três pilares:
1.  **RoPE (Rotary Position Embeddings):** Para melhor generalização temporal.
2.  **MC Dropout:** Para estimar a incerteza epistêmica (o que o modelo não sabe).
3.  **Adaptive Conformal Prediction:** Para garantir estatisticamente a cobertura das previsões (90%), mesmo quando o comportamento dos dados muda (Distribution Shift).

---

In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

try:
    import google.colab
    if not os.path.exists('/content/ufc-easytpp'):
        !git clone https://github.com/hugoramos/ufc-easytpp.git
    project_root = '/content/ufc-easytpp'
except:
    project_root = os.getcwd()
    if os.path.basename(project_root) == 'notebooks':
        project_root = os.path.dirname(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# --- PATCH FP16 ---
import easy_tpp.model.torch_model.torch_baselayer as baselayer
def attention_fixed(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / (d_k ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask > 0, -1e4)
    p_attn = torch.softmax(scores, dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn
baselayer.attention = attention_fixed

# Injetar em RoTHP
try:
    import easy_tpp.model.torch_model.torch_rothp
    easy_tpp.model.torch_model.torch_rothp.attention = attention_fixed
except: pass

from easy_tpp.model.torch_model.torch_rothp import RoTHP

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Hardware: {device}")

### 1. Configuração do Modelo e Dados
Usaremos o dataset `retweet` e configuraremos o RoTHP com Dropout ativado.

In [ ]:
class ThinningConfig:
    def __init__(self):
        self.num_sample = 1
        self.num_exp = 500
        self.over_sample_rate = 5.0
        self.patience_counter = 5
        self.num_samples_boundary = 5
        self.dtime_max = 5.0

class ModelConfig:
    def __init__(self, num_types, pad_id):
        self.hidden_size = 64
        self.time_emb_size = 64
        self.num_layers = 2
        self.num_heads = 4
        self.dropout_rate = 0.2 # Importante para MC Dropout
        self.use_ln = True
        self.num_event_types = num_types
        self.num_event_types_pad = num_types + 1
        self.pad_token_id = pad_id
        self.loss_integral_num_sample_per_step = 20
        self.use_mc_samples = False
        self.gpu = 0 if torch.cuda.is_available() else -1
        self.thinning = ThinningConfig()

def collate_fn_opt(batch_list, pad_id, time_scale):
    batch_size = len(batch_list)
    max_len = max(len(x['time_since_start']) for x in batch_list)
    pad_time = torch.zeros(batch_size, max_len, dtype=torch.float32)
    pad_delta = torch.zeros(batch_size, max_len, dtype=torch.float32)
    pad_type = torch.full((batch_size, max_len), pad_id, dtype=torch.long)
    batch_non_pad_mask = torch.zeros(batch_size, max_len, dtype=torch.float32)
    attention_mask = torch.ones(batch_size, max_len, max_len, dtype=torch.bool)
    causal_mask_base = torch.triu(torch.ones(max_len, max_len, dtype=torch.bool), diagonal=1)
    for i, item in enumerate(batch_list):
        l = len(item['time_since_start'])
        ts = torch.tensor(item['time_since_start'], dtype=torch.float64)
        td = torch.tensor(item['time_since_last_event'], dtype=torch.float64)
        ev = torch.tensor(item['type_event'], dtype=torch.long)
        ts = (ts - ts[0]) / time_scale
        td = td / time_scale
        pad_time[i, :l] = ts.float()
        pad_delta[i, :l] = td.float()
        pad_type[i, :l] = ev
        batch_non_pad_mask[i, :l] = 1.0
        mask_i = causal_mask_base.clone()
        mask_i[:, l:] = True
        mask_i[l:, :] = True
        attention_mask[i] = mask_i
    return (pad_time, pad_delta, pad_type, batch_non_pad_mask, attention_mask)

# Carregar Dados
print("Loading Retweet...")
dataset = load_dataset("easytpp/retweet")
train_data = dataset['train']
dev_data = dataset['validation']
test_data = dataset['test']

all_deltas = []
for item in train_data:
    td = item['time_since_last_event']
    all_deltas.extend([d for d in td if d > 0])
time_scale = np.mean(all_deltas)

num_types = 3
pad_id = 3
collate = lambda x: collate_fn_opt(x, pad_id, time_scale)
train_loader = DataLoader(train_data, batch_size=2048, shuffle=True, collate_fn=collate, num_workers=0)
test_loader = DataLoader(test_data, batch_size=256, shuffle=False, collate_fn=collate, num_workers=0)

### 2. Treinamento do RoTHP
Treinamos o modelo normalmente.

In [ ]:
def train_rothp(epochs=30):
    print(f"\n>>> Treinando RoTHP (Dropout=0.2)...")
    config = ModelConfig(num_types, pad_id)
    model = RoTHP(config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    scaler = torch.amp.GradScaler('cuda')
    
    for epoch in range(1, epochs+1):
        model.train()
        total_loss = 0
        for batch in train_loader:
            batch = [t.to(device, non_blocking=True) for t in batch]
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                loss, num = model.loglike_loss(batch)
                loss = loss / (num + 1e-9)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
        if epoch % 5 == 0:
            print(f"  Ep {epoch}: Loss {total_loss:.4f}")
    return model

model = train_rothp()

### 3. Extração de Incerteza (MC Dropout)
Agora coletamos estatísticas de previsão para o conjunto de teste, rodando o modelo N vezes (MC Samples) para estimar a média e o desvio padrão da previsão.

In [ ]:
def enable_dropout(m):
    """Ativa apenas as camadas de Dropout durante a inferência."""
    for each_module in m.modules():
        if isinstance(each_module, nn.Dropout):
            each_module.train()

def get_mcdropout_predictions(model, loader, n_samples=20):
    """
    Roda o modelo em modo MC Dropout e retorna estatísticas para todo o dataset.
    Retorna: (Média, Desvio Padrão, Valores Reais)
    """
    model.eval()
    enable_dropout(model) # Importante: Dropout ligado no Eval mode
    
    all_means = []
    all_stds = []
    all_targets = []
    
    print(f"Coletando estatísticas MC Dropout (N={n_samples})...")
    with torch.no_grad():
        for batch in tqdm(loader):
            batch = [t.to(device) for t in batch]
            pad_time, pad_delta, pad_type, mask, attn = batch
            
            # Simular N passagens
            batch_samples = []
            for _ in range(n_samples):
                # predict_one_step_at_every_event usa a amostragem interna do modelo
                # para estimar o tempo até o próximo evento (dtime)
                dtimes_pred, _ = model.predict_one_step_at_every_event(batch)
                batch_samples.append(dtimes_pred.cpu().numpy())
            
            # batch_samples shape: [N_samples, Batch, SeqLen]
            batch_samples = np.array(batch_samples)
            
            # Estatísticas sobre as N amostras
            mu = batch_samples.mean(axis=0)
            std = batch_samples.std(axis=0)
            
            # Alvos reais (delta time)
            targets = pad_delta.cpu().numpy()
            mask_cpu = mask.cpu().numpy()
            
            # Flatten e filtrar padding
            # O índice 0 geralmente não tem predição válida anterior em autoregressivos puros,
            # mas predict_one_step_at_every_event já ajusta o shift (retorna predição para próximo).
            # Vamos alinhar com cuidado.
            # pad_delta[:, i] é o tempo real do evento i.
            # predict_one_step_at_every_event retorna em [:, i] a predição feita EM i para i+1? 
            # Olhando o código base: 'time_seq[:, :-1]'... retorna predição baseada nos anteriores.
            # O output tem dimensão seq_len-1 relativo ao input original.
            
            # Vamos assumir o alinhamento padrão: ignorar o último target (pois não prevemos nada depois dele)
            # e ignorar a primeira predição se for garbage. 
            # Simplificando: vamos pegar todos os pares válidos (Pred, Target).
            
            # O EasyTPP geralmente alinha de forma que dtimes_pred[b, i] tenta prever pad_delta[b, i+1]
            # ou algo similar. Vamos verificar shapes.
            # predict retorna [batch, seq_len-1]
            
            # O target correspondente a dtimes_pred[b, i] (que usou eventos 0..i) é pad_delta[b, i+1]
            
            bs, seq_len = targets.shape
            
            # Ajustar targets para alinhar com predições (shift de 1)
            # Targets que estamos tentando prever: do índice 1 até o fim
            real_targets = targets[:, 1:] 
            real_mask = mask_cpu[:, 1:]
            
            # As predições já vêm com tamanho seq_len-1
            # (Verificado no código do EasyTPP: time_seq[:, :-1] ... return dtimes_pred)
            
            for b in range(bs):
                valid_len = int(real_mask[b].sum())
                if valid_len > 0:
                    all_means.extend(mu[b, :valid_len])
                    all_stds.extend(std[b, :valid_len])
                    all_targets.extend(real_targets[b, :valid_len])
                    
    return np.array(all_means), np.array(all_stds), np.array(all_targets)

# Coletar dados do Test Set
means, stds, targets = get_mcdropout_predictions(model, test_loader)

### 4. Simulação de Distribution Shift
Vamos criar um cenário onde o sistema muda de comportamento abruptamente (Shift). Metade dos dados de teste permanece normal, e a outra metade sofre um aumento drástico no tempo entre eventos.

In [ ]:
def induce_shift(targets, start_idx, magnitude=3.0):
    """
    Simula uma mudança no comportamento: os eventos começam a demorar 
    o triplo do tempo previsto a partir de start_idx.
    """
    shifted_targets = targets.copy()
    # A partir do meio, os eventos reais são muito mais longos/curtos que o treino
    shifted_targets[start_idx:] = shifted_targets[start_idx:] * magnitude 
    return shifted_targets

# CRIAR O CENÁRIO DE CAOS
split_point = len(targets) // 2
targets_shifted = induce_shift(targets, split_point, magnitude=3.0) # Shift Brutal de 3x

print(f"Total Eventos: {len(targets)}")
print(f"Ponto de Shift: Evento {split_point}")

### 5. Adaptive Conformal Prediction ("Bulletproof AI")
Implementamos o calibrador dinâmico que ajusta o intervalo de confiança baseado nos erros recentes.

In [ ]:
def conformal_bulletproof(means, stds, targets, alpha=0.1, window_size=100):
    """
    Aplica Conformal Prediction Adaptativo (Rolling Quantile).
    alpha=0.1 significa que queremos 90% de cobertura (Confiança).
    """
    # 1. Calcular Scores de Não-Conformidade (Erro Normalizado)
    # Quão longe o real estava da média, em unidades de desvio padrão?
    scores = np.abs(targets - means) / (stds + 1e-6)
    
    q_hats = [] # Fator de calibração dinâmico
    coverage = [] # Histórico de cobertura
    
    # Valor inicial conservador (aprox 1.96 para normal)
    curr_q = 2.0 
    
    for t in range(len(targets)):
        # -- ADAPTATIVIDADE --
        # Olhamos para os erros dos últimos 'window_size' eventos
        if t > window_size:
            recent_scores = scores[t-window_size : t]
            # Calculamos o quantil (1-alpha) dos erros recentes
            curr_q = np.quantile(recent_scores, 1 - alpha)
            
            # Limites de segurança
            curr_q = min(curr_q, 10.0) 
            curr_q = max(curr_q, 1.0)
            
        q_hats.append(curr_q)
        
        # Verificar se cobriu este evento específico
        lower = means[t] - (curr_q * stds[t])
        upper = means[t] + (curr_q * stds[t])
        is_covered = (targets[t] >= lower) and (targets[t] <= upper)
        coverage.append(is_covered)
        
    return np.array(q_hats), np.array(coverage)

# Rodar a proteção "À Prova de Balas"
q_factors, covered_mask = conformal_bulletproof(means, stds, targets_shifted, alpha=0.1, window_size=200)

# Calcular métricas finais
acc_normal = np.mean(covered_mask[:split_point])
acc_shift = np.mean(covered_mask[split_point:])
print(f"Cobertura antes do Shift: {acc_normal:.2%}")
print(f"Cobertura DURANTE o Shift: {acc_shift:.2%} (Alvo Esperado: ~90%)")

### 6. Visualização da Resiliência
Comparação visual entre a abordagem ingênua (MC Dropout puro) e a abordagem robusta (Adaptive Conformal).

In [ ]:
def plot_resilience(means, stds, targets, q_factors, start=0, end=1000, split_point=None):
    t_idx = np.arange(start, end)
    
    mu = means[start:end]
    sigma = stds[start:end]
    real = targets[start:end]
    qs = q_factors[start:end]
    
    # Intervalo MC Dropout (Padrão/Ingênuo) - +/- 2 desvios
    naive_upper = mu + 2 * sigma
    naive_lower = mu - 2 * sigma
    
    # Intervalo "Bulletproof" (Conformal) - +/- q * desvios
    conf_upper = mu + qs * sigma
    conf_lower = mu - qs * sigma
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), sharex=True)
    
    # --- PLOT 1: O Fracasso do MC Dropout Puro ---
    ax1.set_title("Abordagem Padrão (MC Dropout Naive) - Falha no Shift", fontsize=14, color='red')
    ax1.plot(t_idx, real, 'k.', markersize=3, label='Evento Real', alpha=0.6)
    ax1.plot(t_idx, mu, 'b-', alpha=0.5, label='Predição Média')
    ax1.fill_between(t_idx, naive_lower, naive_upper, color='blue', alpha=0.2, label='Incerteza (Mean ± 2σ)')
    
    if split_point and split_point < end:
        ax1.axvline(x=split_point, color='r', linestyle='--', linewidth=2, label='Início do Shift (Caos)')
        
    ax1.set_ylabel("Tempo até próx. evento")
    ax1.legend(loc='upper left')
    ax1.grid(True, alpha=0.3)

    # --- PLOT 2: A Resiliência do RoTHP Conformal ---
    ax2.set_title("Abordagem Proposta (Adaptive Conformal) - Resiliência Ativa", fontsize=14, color='green')
    ax2.plot(t_idx, real, 'k.', markersize=3, alpha=0.6)
    ax2.plot(t_idx, mu, 'b-', alpha=0.3)
    
    # A área verde se expande automaticamente!
    ax2.fill_between(t_idx, conf_lower, conf_upper, color='green', alpha=0.2, label='Intervalo Garantido (90%)')
    
    # Plotar o fator de correção q_hat num eixo secundário para mostrar a adaptação
    ax2_twin = ax2.twinx()
    ax2_twin.plot(t_idx, qs, color='orange', linestyle=':', linewidth=1.5, label='Fator de Adaptação (q_hat)')
    ax2_twin.set_ylabel("Fator de Expansão da Incerteza", color='orange')
    ax2_twin.tick_params(axis='y', labelcolor='orange')
    
    if split_point and split_point < end:
        ax2.axvline(x=split_point, color='r', linestyle='--', linewidth=2)
        
    ax2.set_ylabel("Tempo até próx. evento")
    ax2.set_xlabel("Índice do Evento")
    ax2.legend(loc='upper left')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Visualizar a transição (pegar 300 eventos antes e 300 depois do shift)
window_view = 600
start_view = max(0, split_point - window_view//2)
end_view = min(len(targets), split_point + window_view//2)

plot_resilience(means, stds, targets_shifted, q_factors, start=start_view, end=end_view, split_point=split_point)